In [ ]:
# Practice Problems Day 8

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 8 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

**Reading:** Geron chapter 2, all of it.

GOALS:

1. Build a `ColumnTransformer` and a `Pipeline` from scratch on new data.
2. See for yourself what happens when a step is left out.
3. Search for settings inside a pipeline, and count the fits before you run them.
4. Open the test set once and report what it says.

**How much of this should you write yourself?** Every problem says. Problem 1 gives you the code with blanks. Problems 2 and 3 list the steps. Problem 4 is by hand. Problem 5 just tells you what to do.

**The data.** Penguins, in `data/penguins.csv`. Three species measured at Palmer Station, Antarctica: bill length and depth, flipper length, body mass, which island they were found on, and their sex. It is small, it is real, and it has holes in it.

**Your task:** predict a penguin's **sex** from its measurements, its species, and its island. Biologists actually want this. The species look alike and sexing a penguin in the field is hard, so a model that does it from measurements is useful.

---

## Problem 1: Look before you build (fill in the blanks)

**What to copy, what to write:** Fill in the blanks marked `____`. The comments say what goes in each one.

**1a.** Fill in the blanks and run it. How many penguins are there, and which columns have missing values?

**1b.** Eleven penguins have no recorded sex. Those rows are dropped in the code below. Why can you not simply fill them in with the most common sex, the way you would fill in a missing bill length?

**1c.** After dropping those eleven, are there any missing values left in your features? Which columns, and how many?

**1d.** What share of the penguins are male? Compare that to the wine and the bank data. Which Day 5 score matters less here than it did there, and why?

In [ ]:
import numpy as np
import pandas as pd

penguins = pd.read_csv("data/penguins.csv")

print("rows and columns:", penguins.shape)
print()
print("missing values per column:")
print(penguins.isna().sum()[penguins.isna().sum() > 0])

# the label is sex, so a penguin with no recorded sex cannot be used
labelled = penguins[penguins["sex"].notna()].copy()     # notna keeps the rows that are NOT missing

y = (labelled["sex"] == "Male").astype(int)             # 1 for male, 0 for female
number_columns = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
text_columns = ["species", "island"]                      # the two non-measurement columns worth using
X = labelled[number_columns + text_columns]

print()
print("penguins with a recorded sex:", len(X))
print("share male:", round(y.mean(), 3))
print("missing values left in X:", X.isna().sum().sum())

**1a.** 344 penguins, 7 columns. Missing: `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm` and `body_mass_g` have 2 each, and `sex` has 11.

**1b.** Because `sex` is the **label**. Filling it in means inventing the answer and then training a model to reproduce your invention. A missing feature can be estimated; a missing label means that row cannot teach you anything.

**1c.** None. The two penguins with missing measurements are among the eleven with no recorded sex, so dropping the unlabelled rows removed them too. (We still build the imputer in Problem 2, because new data will have gaps.)

**1d.** 333 penguins, 50.5 percent male, so the classes are balanced. Average precision mattered on the wine and the bank because positives were rare (13.6 and 11.7 percent). Here plain accuracy is a reasonable headline, and the always-guess-male baseline is only 0.505 instead of 0.86.

---

## Problem 2: Build the pipeline (steps outlined)

**What to copy, what to write:** The steps are comments in the cell. The Day 8 notes have the pattern.

**2a.** Split the test set off (25 percent, stratified, `random_state=42`), then a validation set off the training data the same way. Print the three sizes.

**2b.** Build a `ColumnTransformer` with a number path (impute with the median, then scale) and a text path (impute with the most frequent value, then one-hot encode with `handle_unknown="ignore"`). Fit it on the training features and print how many columns go in and how many come out.

**2c.** Put `LogisticRegression(max_iter=5000)` on the end in a `Pipeline`. Fit it on the training data and print the validation accuracy and average precision.

**2d.** The notes turned 9 columns into 17. Yours turns 6 into something else. Work out on paper where each output column comes from, then check your number.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score

# Step 1 (2a)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42)
print("training:", len(X_train), " validation:", len(X_valid), " test:", len(X_test))

# Step 2 (2b)
number_path = Pipeline([("fill", SimpleImputer(strategy="median")),
                        ("scale", StandardScaler())])
text_path = Pipeline([("fill", SimpleImputer(strategy="most_frequent")),
                      ("dummy", OneHotEncoder(handle_unknown="ignore"))])
preprocessing = ColumnTransformer([("num", number_path, number_columns),
                                   ("cat", text_path, text_columns)])

# Step 3 (2b)
preprocessing.fit(X_train)
print("columns in:", X_train.shape[1], " columns out:", preprocessing.transform(X_train).shape[1])
print(list(preprocessing.get_feature_names_out()))

# Step 4 (2c)
pipe = Pipeline([("prep", preprocessing), ("model", LogisticRegression(max_iter=5000))])
pipe.fit(X_train, y_train)
y_prob = pipe.predict_proba(X_valid)[:, 1]
print("validation accuracy:", round(accuracy_score(y_valid, (y_prob >= 0.5)), 3))
print("validation average precision:", round(average_precision_score(y_valid, y_prob), 3))

**2a.** 186 training, 63 validation, 84 test.

**2b, 2c.** 6 columns in, **10** out. Validation accuracy **0.937**, average precision **0.986**.

**2d.** The four measurements stay four columns. `species` has three values, so three columns. `island` has three values, so three more. 4 plus 3 plus 3 is 10. If you predicted 10 before running it, you understand one-hot encoding!

---

## Problem 3: What each step is worth (steps outlined)

**3a.** Build a second pipeline with **no scaler** in the number path, everything else the same. Fit it and print the validation accuracy and average precision next to the numbers from 2c.

**3b.** Did dropping the scaler hurt? Explain what you see. (Think about what logistic regression does with a feature measured in grams next to one measured in millimetres, and about what `max_iter=5000` is doing for you.)

**3c.** Now try a `RandomForestClassifier(n_estimators=200, random_state=42)` in place of logistic regression, with the full preprocessing. Print its validation numbers.

**3d.** Three sets of numbers now. Say in one sentence which model you would take to the test set, and what would make you change your mind.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Step 1 and 2 (3a)
no_scaler = ColumnTransformer([("num", SimpleImputer(strategy="median"), number_columns),
                               ("cat", text_path, text_columns)])
pipe_noscale = Pipeline([("prep", no_scaler), ("model", LogisticRegression(max_iter=5000))])
pipe_noscale.fit(X_train, y_train)
p = pipe_noscale.predict_proba(X_valid)[:, 1]
print("no scaler:  accuracy", round(accuracy_score(y_valid, (p >= 0.5)), 3),
      " average precision", round(average_precision_score(y_valid, p), 3))

# Step 3 (3c)
pipe_forest = Pipeline([("prep", preprocessing), ("model", RandomForestClassifier(n_estimators=200, random_state=42))])
pipe_forest.fit(X_train, y_train)
p = pipe_forest.predict_proba(X_valid)[:, 1]
print("forest:     accuracy", round(accuracy_score(y_valid, (p >= 0.5)), 3),
      " average precision", round(average_precision_score(y_valid, p), 3))

**3a.** No scaler: accuracy **0.952**, average precision **0.987**. With the scaler it was 0.937 and 0.986.

**3b.** It did not hurt, and it very slightly helped. Here is why: body mass is in thousands of grams while bill depth is around 15 millimetres, so the unscaled features are on wildly different scales, and logistic regression's solver has to work much harder to find the answer. `max_iter=5000` gives it room to get there anyway. On this small, easy dataset it does. Scaling helps the solver find the answer reliably, and it makes the regularization treat every feature the same. With a harder dataset or a smaller `max_iter`, the unscaled version would be the one that fails. If you said the difference is too small to mean anything and gave a reason to keep scaling anyway, that is exactly right.

**3c.** Forest: accuracy **0.921**, average precision **0.986**.

**3d.** All three are the same model as far as this validation set can tell, with 63 penguins in it. Any choice is fine if your reason is something other than the third decimal place: logistic regression is simplest and its coefficients can be read, the forest needs no scaling. What should change your mind is a fairer comparison, which is Problem 4 of the Day 7 homework: repeated cross validation.

---

## Problem 4: By hand, no code

**4a.** You write this grid for a pipeline whose model step is called `model`:

```python
{"model__n_estimators": [100, 200, 400],
 "model__max_depth": [None, 5, 10, 20],
 "model__min_samples_leaf": [1, 3]}
```

With `cv=5`, how many fits does `GridSearchCV` perform? Show the arithmetic.

**4b.** Each fit takes about 0.3 seconds. How long does the search take? Now suppose you add one more setting with 5 choices. How long?

**4c.** Your `ColumnTransformer` names its number path `num`, and inside it the imputer is named `fill`. Write the grid key that would let you search over whether that imputer uses the median or the mean.

**4d.** A classmate fits their `StandardScaler` on the whole dataset, then splits, then searches. Their cross validated score comes out a little higher than yours. Explain what happened, and why their number is not something to be proud of.

**4a.** 3 times 4 times 2 is 24 combinations, times 5 folds, so **120 fits**.

**4b.** 120 times 0.3 seconds is 36 seconds. With another setting of 5 choices: 120 combinations, 600 fits, 180 seconds, so three minutes. Grids multiply! That is why `RandomizedSearchCV` exists.

**4c.** `"prep__num__fill__strategy": ["median", "mean"]`. Reading it right to left: the `strategy` of the step `fill`, inside the path `num`, inside the pipeline step `prep`.

**4d.** Their scaler learned its mean and standard deviation from every row, including the ones that end up in each fold's held out part, so every fold's score includes a little information it should not have had. So their score is too high, they cannot tell by how much, and it will not hold up on new data.

---

## Problem 5: Search, then test once (just the task)

**5a.** Run a `GridSearchCV` over your pipeline on `X_train_full`, searching `model__C` over `[0.01, 0.1, 1, 10, 100]` with `cv=StratifiedKFold(5, shuffle=True, random_state=42)` and `scoring="average_precision"`. Report the best `C`, the best cross validated score, and how many fits that was.

**5b.** Take the winner to the **test set**, once. Print the confusion matrix, the accuracy, and the average precision.

**5c.** Write two or three sentences for a biologist who wants to use this in the field: what the model does, how often it is right, and one thing you would want before trusting it on penguins from a different island.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import confusion_matrix

search = GridSearchCV(
    Pipeline([("prep", preprocessing), ("model", LogisticRegression(max_iter=5000))]),
    {"model__C": [0.01, 0.1, 1, 10, 100]},
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="average_precision")
search.fit(X_train_full, y_train_full)

print("best C:", search.best_params_, " best cross validated AP:", round(search.best_score_, 3))
print("fits:", len(search.cv_results_["params"]) * 5)

y_prob_test = search.best_estimator_.predict_proba(X_test)[:, 1]
y_pred_test = (y_prob_test >= 0.5).astype(int)
print()
print(confusion_matrix(y_test, y_pred_test))
print("test accuracy:", round(accuracy_score(y_test, y_pred_test), 3))
print("test average precision:", round(average_precision_score(y_test, y_prob_test), 3))

**5a.** Best `C` is **1**, cross validated average precision **0.975**, from 5 combinations times 5 folds, so **25 fits**.

**5b.** Test confusion matrix `[[39, 3], [3, 39]]`, accuracy **0.929**, average precision **0.957**. Three females called male and three males called female, out of 84.

**5c.** Your answer will be different, here is one way to say it: "From four body measurements plus species and island, the model tells male from female penguins correctly about 93 times in 100, on 84 birds it had never seen, and it errs about equally in both directions. Before using it on a new island I would want penguins from that island in the training data, because the model has only ever seen three islands and body size differs between colonies." The key pieces are the accuracy, the fact that it was measured on birds the model had never seen, and one honest limitation.